# Example 8: Prommer and Post 2013 (PHT3D Example 10) Dissolution, degradation and geochemical response

The dissolution of an immobile source of non-aequous phase liquid (NAPL) comprised of a petroleum hydrocabon mixture (BTEX) into groundwater is simulated using a first-order, mass-transfer rate expression. The dissolved-phase contaminants are subject to advective-dispersive transport and biodegradation in a heterogeneous unconfined aquifer. Biodegradtion of the dissolved-phase plume is simulated in a two-step partial equilibrium approach (Postma and Jakobsen, 1996). Organic constaminant oxidation is treated as a kinetically-controlled process, whereas the electron accepting process is considered to be close to equilibrium. 

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
import flopy
from mf6rtm import utils, mup3d
from pathlib import Path
import shutil
import re
import difflib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [ ]:
# Phreeqc input file folders
prefix = 'ex8'
dataws = Path.cwd() / 'data'
databasews = Path.cwd() / 'database'
mf6_ws = Path.cwd() / prefix / 'mf6rtm' 
pht3d_ws = Path.cwd() / prefix / 'pht3d'

## Flow and Transport Setup

In [ ]:
# Load MODFLOW model
mf2k = flopy.modflow.Modflow.load('ex10_pht3d.nam', model_ws=pht3d_ws, version='mf2k')

# Temporal dicretization
time_units = 'days'
nper = mf2k.dis.nper
nstp = mf2k.dis.nstp.array[0]
perlen = mf2k.dis.perlen.array[0]
perioddata = [[perlen, nstp, 1.] for kper in range(nper)]

# Spatial discretization
length_units = 'meters'
nlay, nrow, ncol = mf2k.dis.nlay, mf2k.dis.nrow, mf2k.dis.ncol
delr, delc = mf2k.dis.delr.array[0], mf2k.dis.delc.array[0]
top = mf2k.dis.top.array
botm = mf2k.dis.botm.array
idomain = np.ones((nlay, nrow, ncol), dtype=int)

# Hydraulic properties
por = 0.3  # Porosity
k11 = k33 = mf2k.lpf.hk.array

# ic
strt = mf2k.bas6.strt.array

# constant heads 
chd_in = [[(0,i,0),strt[0,0,0]] for i in range(nrow)]
chd_out = [[(0,i,ncol-1),strt[0,0,ncol-1]] for i in range(nrow)]
chd_spd = chd_in + chd_out

# Confined aquifer
icelltype = iconvert = 0

# NAPL IC
benz_ic = np.loadtxt(pht3d_ws / 'Pht3d.btn', skiprows=35, max_rows=40)
assert benz_ic.shape == top.shape

tol_ic = np.loadtxt(pht3d_ws / 'Pht3d.btn', skiprows=76, max_rows=40)
assert tol_ic.shape == top.shape

# Dispersivity
alh = .5 # m
ath1 = .1 # m

In [ ]:
perioddata

## Geochemistry Setup

In [ ]:
solutionsdf = pd.read_csv(dataws/ f"{prefix}_solutions.csv", index_col=0)
solutions = utils.solution_df_to_dict(solutionsdf)

# instantiate solutions class
solution = mup3d.Solutions(solutions)
sol_ic = np.ones_like(idomain)
sol_ic[0][benz_ic!=0] = 2 # napl source
solution.set_ic(sol_ic)

# kinetics
kin_df = pd.read_csv(dataws / f"{prefix}_kinetic_phases.csv")
kinetics=utils.parse_kinetics_dataframe(kin_df)
num = list(kinetics.keys())[0]

# let's add the phase reaction formulas
benz_degradation = 'Benz -1.0 C6H6 1.0'
kinetics[num]['Benz']['formula'] = benz_degradation
tolu_degradation = 'Tolu -1.0 C7H8 1.0'
kinetics[num]['Tolu']['formula'] = tolu_degradation
ethy_degradation = 'Ethy -1.0 C8H10 1.0'
kinetics[num]['Ethy']['formula'] = ethy_degradation
xyl_degradation = 'Xyl -1.0 C8H10 1.0'
kinetics[num]['Xyl']['formula'] = xyl_degradation
benz_dissolution = 'Benznapl -1.0 Benz   1.0'
kinetics[num]['Benznapl']['formula'] = benz_dissolution
tolu_dissolution = 'Tolunapl -1.0 Tolu   1.0'
kinetics[num]['Tolunapl']['formula'] = tolu_dissolution
ethy_dissolution = 'Ethynapl -1.0  Ethy   1.0'
kinetics[num]['Ethynapl']['formula'] = ethy_dissolution
xyl_dissolution = 'Xylnapl  -1.0  Xyl    1.0'
kinetics[num]['Xylnapl']['formula'] = xyl_dissolution
# instantiate kinetic phases class
kinetics = mup3d.KineticPhases(kinetics)

kinetics.set_ic(num) 

# equilibrium phases
eq_df = pd.read_csv(os.path.join(dataws /f"{prefix}_equilibrium_phases.csv"))
eq_phases = utils.parse_equilibriums_dataframe(eq_df,
                                               columns = ['phase', 'sat_index', 'conc_mol_l', 'num'])

for key, value in eq_phases.items():
    for k, v in value.items():
        v['m0'] = utils.concentration_volbulk_to_volwater(v['m0'], por)

# instantiate equilibirum phases class
eq_phases = mup3d.EquilibriumPhases(eq_phases)
eq_ic = np.ones_like(idomain)
eq_phases.set_ic(eq_ic)

# instantiate model class
model = mup3d.Mup3d(prefix, solution, nlay, nrow, ncol)

# set model workspace
if os.path.exists(mf6_ws):
    shutil.rmtree(mf6_ws)
model.set_wd(mf6_ws)

# set database
database = databasews / f'{prefix}_pht3d_datab.dat'
model.set_database(database)

model.set_phases(kinetics)
model.set_phases(eq_phases)

# phreeqc output
postfix = dataws / f'{prefix}_postfix.phqr'
model.set_postfix(postfix)

# Transport only excess H2O, H, and O
model.set_componenth2o(True)

print('initializing mf6rtm model..\n')
model.initialize()

benznapl_ic = model.sconc['Benznapl']!=0
napl_ic = benz_ic!=0
assert benznapl_ic.sum() == napl_ic.sum(), 'check NAPL location'

# Inflow chemistry
chdchem = mup3d.ChemStress('chdin')
sol_chem = [1] # equal to solution IC, in this case..
chdchem.set_spd(sol_chem)
model.set_chem_stress(chdchem)

In [ ]:
# add concentrations to CHD spd
for kper in range(len(chd_spd)):
    chd_spd[kper].extend(model.chdin.data[0])

In [ ]:
def build_mf6(mup3d, nper, perioddata, len_units, time_units, nlay, nrow, ncol,
              delr, delc, top, botm, idomain, chdspd, por, k11, k33, icelltype,
              strt, alh, ath1):

    #####################        GWF model           #####################
    gwfname = 'gwf' # must be called gwf
    sim_ws = mup3d.wd
    exe_path = os.path.join(sim_ws, 'mf6')
    sim = flopy.mf6.MFSimulation(sim_name=mup3d.name, sim_ws=sim_ws,
                                 exe_name=exe_path, version='mf6')

    # Instantiating MODFLOW 6 time discretization
    flopy.mf6.ModflowTdis(sim, nper=nper, perioddata=perioddata,
                          time_units=time_units)

    # Instantiating MODFLOW 6 groundwater flow model
    gwf = flopy.mf6.ModflowGwf(sim, modelname=gwfname, save_flows=True, model_nam_file=f"{gwfname}.nam")

    # Instantiating MODFLOW 6 solver for flow model
    imsgwf = flopy.mf6.ModflowIms(
        sim,
        complexity="simple",
        print_option="summary",
        outer_dvclose=1e-3,
        outer_maximum=200,
        backtracking_number=20,
        backtracking_tolerance=1.2,
        backtracking_reduction_factor=0.7,
        backtracking_residual_limit=100,
        under_relaxation="none",
        inner_maximum=500,
        inner_dvclose=1e-5,
        rcloserecord=1e-3,
        linear_acceleration="bicgstab",
        scaling_method="none",
        reordering_method="none",
        filename=f"{gwfname}.ims")
    
    sim.register_ims_package(imsgwf, [gwf.name])

    # Instantiating MODFLOW 6 discretization package
    dis = flopy.mf6.ModflowGwfdis(
        gwf,
        length_units=len_units,
        nlay=nlay,
        nrow=nrow,
        ncol=ncol,
        delr=delr,
        delc=delc,
        top=top,
        botm=botm,
        idomain=idomain,
        filename=f"{gwfname}.dis",
    )
    dis.set_all_data_external()

    # Instantiating MODFLOW 6 node-property flow package
    npf = flopy.mf6.ModflowGwfnpf(
        gwf,
        save_flows=False,
        save_saturation=False,
        icelltype=icelltype,
        k=k11,
        k33=k33,
        save_specific_discharge=False,
        filename=f"{gwfname}.npf",
    )
    npf.set_all_data_external()

    sto = flopy.mf6.ModflowGwfsto(gwf, ss=np.full(idomain.shape,1e-6),
                                  sy=np.full(idomain.shape, 0.2),
                                  iconvert=[0], steady_state={0:True})

    # Instantiating MODFLOW 6 initial conditions package for flow model
    flopy.mf6.ModflowGwfic(gwf, strt=strt)
    
    # Instantiating MODFLOW 6 constant head package
    chd = flopy.mf6.ModflowGwfchd(
        gwf,
        maxbound=len(chdspd),
        stress_period_data=chdspd,
        auxiliary=mup3d.components,
        save_flows=False,
        pname="chd",
    )
    chd.set_all_data_external()

    # Instantiating MODFLOW 6 output control package for flow model
    oc_gwf = flopy.mf6.ModflowGwfoc(
        gwf,
        head_filerecord=f"{gwfname}.hds",
        budget_filerecord=f"{gwfname}.cbb",
        budgetcsv_filerecord='budget.csv',
        # headprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 6, "GENERAL")],
        saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
        # printrecord=[("HEAD", "LAST"), ("BUDGET", "LAST")]
)

    #####################           GWT model          #####################
    for c in mup3d.components:
        print(f'Setting model for component: {c}')
        gwtname = c
        
        # Instantiating MODFLOW 6 groundwater transport package
        gwt = flopy.mf6.MFModel(
            sim,
            model_type="gwt6",
            modelname=gwtname,
            model_nam_file=f"{gwtname}.nam"
        )

        # create iterative model solution and register the gwt model with it
        print('--- Building IMS package ---')
        imsgwt = flopy.mf6.ModflowIms(
            sim,
            # complexity="simple",
            print_option="all",
            outer_dvclose=1e-3,
            outer_maximum=200,
            under_relaxation="none",
            inner_maximum=500,
            inner_dvclose=1e-5,
            rcloserecord=1e-3,
            linear_acceleration="bicgstab",
            scaling_method="none",
            reordering_method="none",
            filename=f"{gwtname}.ims",
        )
        sim.register_ims_package(imsgwt, [gwt.name])

        print('--- Building DIS package ---')
        dis = gwf.dis

        # create grid object
        dis = flopy.mf6.ModflowGwtdis(
            gwt,
            length_units=length_units,
            nlay=nlay,
            nrow=nrow,
            ncol=ncol,
            delr=delr,
            delc=delc,
            top=top,
            botm=botm,
            idomain=idomain,
            filename=f"{gwtname}.dis",
        )
        # dis.set_all_data_external()
         
        ic = flopy.mf6.ModflowGwtic(gwt, strt=mup3d.sconc[c], filename=f"{gwtname}.ic")
        # ic.set_all_data_external()
        
        # Instantiating MODFLOW 6 transport source-sink mixing package
        sourcerecarray = ['chd', 'aux', f'{c}']
        ssm = flopy.mf6.ModflowGwtssm(
            gwt, 
            save_flows=True,
            sources=sourcerecarray, 
            # print_flows=True,
            filename=f"{gwtname}.ssm"
        )
        # ssm.set_all_data_external()
        
        if 'napl' not in c:            
            # Instantiating MODFLOW 6 transport adv package
            print('--- Building ADV package ---')
            adv = flopy.mf6.ModflowGwtadv(
                gwt,
                scheme="tvd"
                )
    
            # Instante MODFLOW 6 transport dispersion package
            alpha_l = np.ones(shape=(nlay, nrow, ncol))*alh  # Longitudinal dispersivity (m)
            ath1 = np.ones(shape=(nlay, nrow, ncol))*ath1 # Transverse horizontal dispersivity (m)
            # atv = np.ones(shape=(nlay, nrow, ncol))*atv   # Transverse vertical dispersivity (m)
    
            print('--- Building DSP package ---')
            dsp = flopy.mf6.ModflowGwtdsp(
                gwt,
                xt3d_off=True,
                alh=alh,  # Longitudinal dispersivity (m)
                ath1=ath1, # Transverse horizontal dispersivity (m)
                # atv = atv,  # Transverse vertical dispersivity (m)
                # diffc = diffc,
                filename=f"{gwtname}.dsp"
                )
            # dsp.set_all_data_external()
    
            # Instantiating MODFLOW 6 transport mass storage package (formerly "reaction" package in MT3DMS)
            print('--- Building MST package ---')
    
            first_order_decay = None

        mst = flopy.mf6.ModflowGwtmst(
            gwt,
            porosity=por,
            first_order_decay=first_order_decay,
            filename=f"{gwtname}.mst"
            )
        # mst.set_all_data_external()

        print('--- Building OC package ---')

        # Instantiating MODFLOW 6 transport output control package
        oc_gwt = flopy.mf6.ModflowGwtoc(
            gwt,
            budget_filerecord=f"{gwtname}.cbb",
            concentration_filerecord=f"{gwtname}.ucn",
            concentrationprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 10, "GENERAL")
                                        ],
            saverecord=[("CONCENTRATION", "ALL"), 
                        ("BUDGET", "ALL")
                        ]
        )
            # printrecord=[("CONCENTRATION", "ALL"), 
            #                ("BUDGET", "ALL")
            #                ]

        # Instantiating MODFLOW 6 flow-transport exchange mechanism
        flopy.mf6.ModflowGwfgwt(
            sim,
            exgtype="GWF6-GWT6",
            exgmnamea=gwfname,
            exgmnameb=gwtname,
            filename=f"{gwtname}.gwfgwt",
        )

    sim.write_simulation()
    utils.prep_bins(sim_ws, src_path=os.path.join('..','bin'),
                    get_only=['mf6', 'libmf6'])

    return sim

# Run MF6 model. If succesful, run MF6RTM.

In [ ]:
print('building mf6 model..\n')
sim = build_mf6(model, nper, perioddata, length_units, time_units, nlay, nrow,
                ncol, [delr], [delc], top, botm, idomain, chd_spd, por, k11, k33,
                icelltype, strt, alh, ath1)

print('running mf6 model without reactions..\n')
success = model.run(reactive = False)

if success:
    print('running mf6rtm model..\n')
    print("MODFLOW 6 simulation ran successfully.")
    model.run()
else:
    print("MODFLOW 6 simulation failed.")

## Process results

In [ ]:
## Load mf6 flow model
sim = flopy.mf6.MFSimulation.load(sim_ws=mf6_ws, verbosity_level=0)
gwf = sim.get_model('gwf')
mg = gwf.modelgrid

In [ ]:
## Read mf6rtm results punched out by PHREEQC
df = pd.read_csv(mf6_ws / 'sout.csv')
df = df.loc[df.day==500]
df.loc[:,'row'] = df.cell.apply(lambda x: mg.get_lrc(int(x))[0][1])
df.loc[:,'column'] = df.cell.apply(lambda x: mg.get_lrc(int(x))[0][2])

mf6rtm_conc = {c:np.full((nrow,ncol),np.nan) for c in df.columns[2:-2]}
for c in df.columns[2:-2]:
    mf6rtm_conc[c][df.row,df.column] = df[c]

## Read MF6 UCN files
ucn_files = [f for f in os.listdir(mf6_ws) if f.lower().endswith('.ucn')]
ucndict_mf6 = {}
for uf in ucn_files:
    ucnobj = flopy.utils.HeadFile(mf6_ws / uf, text="concentration")
    ucn = ucnobj.get_alldata()
    ucndict_mf6[f'{uf.split(".")[0]}'] = ucn
timesmf6 = ucnobj.get_times()

## Read PHT3D UCN files
ucn_files = [f for f in os.listdir(pht3d_ws) if f.lower().endswith('.ucn')]

# get file that ends in py
pht3dpy = [f for f in os.listdir(pht3d_ws) if f.endswith('py')]
# read pht3dpy file
pht3dpy = pht3d_ws / pht3dpy[0]  
ucndict_pht3d = {}
with open(pht3dpy, 'r') as f: 
    for l in f:
        n = re.findall(r'\d+', l.split()[-1])[-1]
        if int(n) == 15:
            ucndict_pht3d['Methane'] = f"PHT3D{n}.UCN"
        ucndict_pht3d[l.split()[0].replace('_', "")] = f"PHT3D{n}.UCN"
        
for k,v in ucndict_pht3d.items():
    ucnobj = flopy.utils.HeadFile(pht3d_ws / v, text="concentration")
    ucn = ucnobj.get_alldata()
    ucndict_pht3d[k] = ucn
timespht3d = ucnobj.get_times()

In [ ]:
# helper function to match component names
def find_closest_match(query, dictionary):
    closest_match = difflib.get_close_matches(query, dictionary.keys(), n=1)
    if closest_match:
        return closest_match[0]
    else:
        return None

In [ ]:
# Component names
plt_dict = {
    1:{'Benz':'Benzene', 'Tolu':'Toluene'},
    2:{'O':'Oxygen', 
       ('NO3', 'N5'):'Nitrate'},
    3:{('SO4', 'S6'):'Sulfate',
       'C4':'DIC'},
    4:{'eq_Pyrite':'Pyrite', 'Calcite':'Calcite'},
    5:{'H':'pH', 'Ca':'Ca'},
    6:{'eq_Goethite':'Goethite', ('C_4','Methane'):'Methane'}
    }

napl_dict = {'Benznapl':'Benzene', 
             'Tolunapl':'Toluene'} 

## Visualize dissolved-phase plumes, mineral phases, and NAPL concentrations

In [ ]:
## Plot dissolved phase plumes
for key,names_dict in plt_dict.items():
    cm = 'plasma'
    if len(names_dict)>1:
        fig, axes = plt.subplots(2, 2, figsize=(16, 8))
    else:
        fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    fig.suptitle('Dissolved-Phase Plumes - 500 days',
                 fontsize=18)
    ct = 0
    names = []
    for c,name in names_dict.items(): 

        print(name)
        if name in ['Benzene', 'Toluene']:
            mf6_c = find_closest_match(c, ucndict_mf6)
            pht3d_c = find_closest_match(c, ucndict_pht3d)
        elif isinstance(c, tuple):
            mf6_c = find_closest_match(c[0], mf6rtm_conc)
            pht3d_c = find_closest_match(c[1], ucndict_pht3d)
        else:   
            mf6_c = find_closest_match(c, mf6rtm_conc)
            pht3d_c = find_closest_match(c, ucndict_pht3d)
        print(mf6_c, ' mf6')    
        print(pht3d_c, 'pht3d', '\n')
        names.append(name)
        
        if name in ['Benzene', 'Toluene']:    
            mf6_conc = ucndict_mf6[mf6_c][49][0]/1000 # from m3 to L
        else:
            mf6_conc = mf6rtm_conc[mf6_c]
        mf6_conc[mf6_conc<=0] = 0
        
        pht3d_conc = ucndict_pht3d[pht3d_c][0][0] 
        pht3d_conc[pht3d_conc<=0] = 0
    
        # Plot mf6
        ax1 = axes.flatten()[ct]
        ax1.set_title(f'{name} (MF6RTM)', fontsize=12)
    
        mv1 = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax1)
        mv1.plot_grid(alpha=0.5)
        cs1 = mv1.plot_array(mf6_conc, cmap=cm)
        mv1.plot_bc('CHD', color='cyan')
        
        divider1 = make_axes_locatable(ax1)
        cax1 = divider1.append_axes("right", size="2%", pad=0.05)
        cbar1 = plt.colorbar(cs1, cax=cax1)
        cbar1.set_label('(mol/L)',  rotation=270, labelpad=15, fontsize=12)
        
        # Plot pht3d
        ax2 = axes.flatten()[ct+1]
        ax2.set_title(f'{name} (PHT3D)', fontsize=12)
    
        mv2 = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax2)
        mv2.plot_grid(alpha=0.5)
        cs2 = mv2.plot_array(pht3d_conc, cmap=cm)
        mv2.plot_bc('CHD', color='cyan')
        
        divider2 = make_axes_locatable(ax2)
        cax2 = divider2.append_axes("right", size="2%", pad=0.05)
        cbar2 = plt.colorbar(cs2, cax=cax2)
        cbar2.set_label('(mol/L)',  rotation=270, labelpad=15, fontsize=12)
        
        ct += 2
    
    fig.tight_layout()
    plt.show()
    if len(names)>1:
        fig.savefig(f'ex8_{names[0]}_{names[1]}_500_days.png', dpi=300)
    else:
        fig.savefig(f'ex8_{names[0]}_500_days.png', dpi=300)
    plt.close()

In [ ]:
## Plot NAPL concentrations
cm = 'cool'
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle('NAPL concentrations - 500 days',
             fontsize=18)
ct = 0
for c,name in napl_dict.items(): 
    mf6_c = find_closest_match(c, ucndict_mf6)
    pht3d_c = find_closest_match(c, ucndict_pht3d)
    
    mf6_conc = ucndict_mf6[mf6_c][49][0]/1000 # from m3 to L
    mf6_conc[mf6_conc<=0] = np.nan
    
    pht3d_conc = ucndict_pht3d[pht3d_c][0][0] 
    pht3d_conc[pht3d_conc<=0] = np.nan

    # Plot mf6
    ax1 = axes.flatten()[ct]
    ax1.set_title(f'{name} (MF6RTM)', fontsize=12)

    mv1 = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax1)
    mv1.plot_grid(alpha=0.5)
    cs1 = mv1.plot_array(mf6_conc, cmap=cm)
    mv1.plot_bc('CHD', color='blue')
    
    divider1 = make_axes_locatable(ax1)
    cax1 = divider1.append_axes("right", size="2%", pad=0.05)
    cbar1 = plt.colorbar(cs1, cax=cax1)
    cbar1.set_label('(mol/L)',  rotation=270, labelpad=15, fontsize=12)
    
    # Plot pht3d
    ax2 = axes.flatten()[ct+1]
    ax2.set_title(f'{name} (PHT3D)', fontsize=12)

    mv2 = flopy.plot.PlotMapView(model=gwf, layer=0, ax=ax2)
    mv2.plot_grid(alpha=0.5)
    cs2 = mv2.plot_array(pht3d_conc, cmap=cm)
    mv2.plot_bc('CHD', color='blue')
    
    divider2 = make_axes_locatable(ax2)
    cax2 = divider2.append_axes("right", size="2%", pad=0.05)
    cbar2 = plt.colorbar(cs2, cax=cax2)
    cbar2.set_label('(mol/L)',  rotation=270, labelpad=15, fontsize=12)
    
    ct += 2

fig.tight_layout()
plt.show()
fig.savefig('ex8_napl_500_days.png', dpi=300)
plt.close()